# Anime pass GPU server

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Wait for the last cell to print a URL and a token (about 5-8 min the first time: it downloads ~9 GB of models).
3. Paste both into the chat. Keep this tab open while the panels render.

In [ ]:
!nvidia-smi -L
!pip -q install diffusers==0.41.0 transformers accelerate safetensors opencv-python-headless

In [ ]:
import os, json
FILES = json.loads('{"animescene/__init__.py": "", "animescene/diffusion.py": "\\"\\"\\"SDXL + ControlNet-Union sampler with regional (per-character) prompting.\\n\\nBuilt directly on diffusers *components* (UNet, ControlNetUnion, VAE, CLIPs)\\nwith our own denoising loop, because the stock pipelines can\'t do what the\\nstage gives us:\\n\\n* **Regional cross-attention**: every character\'s prompt only speaks to the\\n  latent pixels its 3D silhouette owns (plus a shared scene prompt mixed in\\n  everywhere), so four fighters keep four distinct designs and palettes.\\n* **Multi-condition ControlNet-Union**: OpenPose + depth (+ optional normal)\\n  rendered from the same 3D scene, so the pose and the occlusion order agree.\\n* **txt2img, img2img and masked inpaint** in the same loop, used for the\\n  base pass, the hires pass and the per-character detail passes.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\nimport os\\nimport time\\nfrom dataclasses import dataclass, field\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom PIL import Image\\n\\nCONTROL_MODES = {\\"openpose\\": 0, \\"depth\\": 1, \\"hed\\": 2, \\"scribble\\": 2, \\"canny\\": 3, \\"lineart\\": 3,\\n                 \\"normal\\": 4, \\"segment\\": 5, \\"tile\\": 6, \\"repaint\\": 7}\\nDEFAULT_MODELS = os.environ.get(\\"ANIMESCENE_MODELS\\", \\"/home/user/models\\")\\n\\n\\n# ------------------------------------------------------------------ regional attention\\n@dataclass\\nclass RegionState:\\n    active: bool = False\\n    contexts: list = field(default_factory=list)  # per region (1, L, 2048) cond embeddings\\n    masks: torch.Tensor | None = None  # (R, h, w) latent-resolution soft masks\\n    base_weight: float = 0.3\\n    cache: dict = field(default_factory=dict)\\n\\n    def masks_for(self, n_tokens: int, dtype) -> torch.Tensor:\\n        \\"\\"\\"(R, N, 1) masks resampled to an attention layer\'s token grid.\\"\\"\\"\\n        if n_tokens in self.cache:\\n            return self.cache[n_tokens]\\n        R, h, w = self.masks.shape\\n        for k in range(6):\\n            hh, ww = math.ceil(h / 2 ** k), math.ceil(w / 2 ** k)\\n            if hh * ww == n_tokens:\\n                break\\n        else:\\n            raise RuntimeError(f\\"cannot map {n_tokens} tokens to latent {h}x{w}\\")\\n        m = F.interpolate(self.masks[None].float(), size=(hh, ww), mode=\\"area\\")[0]\\n        m = m.reshape(R, -1, 1).to(dtype)\\n        self.cache[n_tokens] = m\\n        return m\\n\\n\\ndef _attend(attn, hidden, ctx, bias=None):\\n    q = attn.to_q(hidden)\\n    k = attn.to_k(ctx)\\n    val = attn.to_v(ctx)\\n    B, N, _ = q.shape\\n    hd = k.shape[-1] // attn.heads\\n    q = q.view(B, -1, attn.heads, hd).transpose(1, 2)\\n    k = k.view(k.shape[0], -1, attn.heads, hd).transpose(1, 2)\\n    val = val.view(val.shape[0], -1, attn.heads, hd).transpose(1, 2)\\n    if k.shape[0] != B:\\n        k, val = k.expand(B, -1, -1, -1), val.expand(B, -1, -1, -1)\\n    o = F.scaled_dot_product_attention(q, k, val, attn_mask=bias)\\n    o = o.transpose(1, 2).reshape(B, N, attn.heads * hd).to(q.dtype)\\n    o = attn.to_out[0](o)\\n    return attn.to_out[1](o)\\n\\n\\nclass RegionalCrossAttn:\\n    \\"\\"\\"Cross-attention processor: batch is [uncond, cond]; the cond half is a mask-weighted\\n    mixture of the base prompt and one prompt per region (the last region = background).\\"\\"\\"\\n\\n    def __init__(self, state: RegionState):\\n        self.state = state\\n\\n    def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, temb=None,\\n                 **kwargs):\\n        if encoder_hidden_states is None:\\n            encoder_hidden_states = hidden_states\\n        out = _attend(attn, hidden_states, encoder_hidden_states)\\n        st = self.state\\n        if not st.active or not st.contexts:\\n            return out\\n        B = hidden_states.shape[0]\\n        half = B // 2\\n        hc = hidden_states[half:]\\n        masks = st.masks_for(hc.shape[1], hc.dtype)  # (R, N, 1)\\n        b = st.base_weight\\n        leftover = (1 - masks.sum(0)).clamp(min=0)[None]  # pixels no region owns -> base prompt\\n        mix = out[half:] * (b + (1 - b) * leftover)\\n        for r, ctx in enumerate(st.contexts):\\n            mix = mix + (1 - b) * masks[r][None] * _attend(attn, hc, ctx.to(hc.dtype))\\n        return torch.cat([out[:half], mix], 0)\\n\\n\\n# ------------------------------------------------------------------ reference attention\\n@dataclass\\nclass Reference:\\n    \\"\\"\\"A character identity anchor: its sheet image, its prompt, and where it applies.\\"\\"\\"\\n    image: Image.Image\\n    prompt: str\\n    mask: np.ndarray | None = None  # HxW in output pixels; None = whole image\\n    weight: float = 1.0\\n\\n\\n@dataclass\\nclass RefState:\\n    mode: str = \\"off\\"  # off | write | read\\n    banks: list = field(default_factory=list)  # per reference: {layer: (1, N_r, C)}\\n    current: int = 0\\n    masks: torch.Tensor | None = None  # (R, h, w) latent-resolution; rows of ones = everywhere\\n    weights: list = field(default_factory=list)\\n    style_fidelity: float = 0.5\\n    cache: dict = field(default_factory=dict)\\n\\n    def bias(self, n_tokens: int, ref_lens: list, dtype) -> torch.Tensor:\\n        \\"\\"\\"Additive attention bias (N, N + sum(ref_lens)): each pixel may attend to its own\\n        character\'s reference tokens only (log-weighted), never to another character\'s.\\"\\"\\"\\n        key = (n_tokens, tuple(ref_lens))\\n        if key in self.cache:\\n            return self.cache[key]\\n        R, h, w = self.masks.shape\\n        for k in range(6):\\n            hh, ww = math.ceil(h / 2 ** k), math.ceil(w / 2 ** k)\\n            if hh * ww == n_tokens:\\n                break\\n        else:\\n            raise RuntimeError(f\\"cannot map {n_tokens} tokens to latent {h}x{w}\\")\\n        m = F.interpolate(self.masks[None].float(), size=(hh, ww), mode=\\"area\\")[0].reshape(R, -1)\\n        cols = [torch.zeros(n_tokens, n_tokens)]\\n        for r, L in enumerate(ref_lens):\\n            wgt = torch.clamp(m[r] * self.weights[r], min=1e-6).log()\\n            wgt[m[r] < 0.05] = float(\\"-inf\\")\\n            cols.append(wgt[:, None].expand(n_tokens, L))\\n        b = torch.cat(cols, 1).to(dtype)\\n        self.cache[key] = b\\n        return b\\n\\n\\nclass RefSelfAttn:\\n    \\"\\"\\"Self-attention that, in \\"write\\" mode, records a reference image\'s features and, in \\"read\\"\\n    mode, lets each pixel also attend to the features of its own character\'s reference sheet.\\"\\"\\"\\n\\n    def __init__(self, state: RefState, name: str):\\n        self.state, self.name = state, name\\n\\n    def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, temb=None, **kw):\\n        st = self.state\\n        if st.mode == \\"write\\":\\n            st.banks[st.current][self.name] = hidden_states.detach()\\n            return _attend(attn, hidden_states, hidden_states)\\n        if st.mode != \\"read\\" or not st.banks:\\n            return _attend(attn, hidden_states, hidden_states)\\n        B, N, _ = hidden_states.shape\\n        refs = [b[self.name].to(hidden_states.dtype) for b in st.banks]\\n        ctx = torch.cat([hidden_states] + [r.expand(B, -1, -1) for r in refs], 1)\\n        bias = st.bias(N, [r.shape[1] for r in refs], hidden_states.dtype)\\n        out = _attend(attn, hidden_states, ctx, bias)\\n        if B >= 2 and st.style_fidelity > 0:  # uncond half: partly plain, keeps the CFG contrast\\n            half = B // 2\\n            plain = _attend(attn, hidden_states[:half], hidden_states[:half])\\n            out = torch.cat([st.style_fidelity * plain + (1 - st.style_fidelity) * out[:half], out[half:]])\\n        return out\\n\\n\\ndef _fit_area(size, area):\\n    \\"\\"\\"Scale (w, h) to about `area` pixels, multiples of 64.\\"\\"\\"\\n    w, h = size\\n    k = math.sqrt(area / (w * h))\\n    return max(64, int(round(w * k / 64)) * 64), max(64, int(round(h * k / 64)) * 64)\\n\\n\\n# ------------------------------------------------------------------ engine\\nclass Engine:\\n    def __init__(self, models: str = DEFAULT_MODELS, dtype=None, threads: int | None = None, device: str | None = None):\\n        device = device or (\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n        dtype = dtype or (torch.float16 if device == \\"cuda\\" else torch.bfloat16)\\n        self.dev = torch.device(device)\\n        from diffusers import AutoencoderKL, ControlNetUnionModel, DPMSolverMultistepScheduler, \\\\\\n            UNet2DConditionModel\\n        from transformers import CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer\\n\\n        if device == \\"cpu\\":\\n            torch.set_num_threads(threads or os.cpu_count() or 4)\\n        t0 = time.time()\\n        base = os.path.join(models, \\"sdxl\\")\\n        self.dtype = dtype\\n        self.tok1 = CLIPTokenizer.from_pretrained(base, subfolder=\\"tokenizer\\")\\n        self.tok2 = CLIPTokenizer.from_pretrained(base, subfolder=\\"tokenizer_2\\")\\n        self.te1 = CLIPTextModel.from_pretrained(base, subfolder=\\"text_encoder\\", torch_dtype=dtype).eval()\\n        self.te2 = CLIPTextModelWithProjection.from_pretrained(base, subfolder=\\"text_encoder_2\\",\\n                                                               torch_dtype=dtype).eval()\\n        self.unet = UNet2DConditionModel.from_pretrained(base, subfolder=\\"unet\\", torch_dtype=dtype).eval()\\n        vae_dir = os.path.join(models, \\"vae_fix\\")\\n        self.vae = AutoencoderKL.from_pretrained(vae_dir if os.path.isdir(vae_dir) else base,\\n                                                 **({} if os.path.isdir(vae_dir) else {\\"subfolder\\": \\"vae\\"}),\\n                                                 torch_dtype=torch.float32).eval()\\n        self.vae.enable_tiling()\\n        cn_dir = os.path.join(models, \\"controlnet_union\\")\\n        self.controlnet = ControlNetUnionModel.from_pretrained(cn_dir, torch_dtype=dtype).eval() \\\\\\n            if os.path.isdir(cn_dir) else None\\n        self.sched_config = DPMSolverMultistepScheduler.load_config(base, subfolder=\\"scheduler\\")\\n        self.region = RegionState()\\n        self.ref = RefState()\\n        procs = {}\\n        for name, proc in self.unet.attn_processors.items():\\n            if name.endswith(\\"attn2.processor\\"):\\n                procs[name] = RegionalCrossAttn(self.region)\\n            elif name.endswith(\\"attn1.processor\\"):\\n                procs[name] = RefSelfAttn(self.ref, name)\\n            else:\\n                procs[name] = proc\\n        self.unet.set_attn_processor(procs)\\n        for m in (self.te1, self.te2, self.unet, self.vae, self.controlnet):\\n            if m is not None:\\n                m.to(self.dev)\\n        self.loaded_loras: dict = {}\\n        print(f\\"[engine] loaded in {time.time() - t0:.0f}s\\", flush=True)\\n\\n    # -------------------------------------------------------------- prompts\\n    @torch.no_grad()\\n    def encode(self, prompt: str, chunks: int | None = None):\\n        \\"\\"\\"Long-prompt SDXL encoding (A1111-style 75-token chunks). Returns (emb, pooled, n_chunks).\\"\\"\\"\\n        ids1 = self.tok1(prompt, add_special_tokens=False).input_ids\\n        ids2 = self.tok2(prompt, add_special_tokens=False).input_ids\\n        n = max(1, math.ceil(max(len(ids1), len(ids2)) / 75))\\n        if chunks is not None:\\n            if n > chunks:\\n                ids1, ids2, n = ids1[:75 * chunks], ids2[:75 * chunks], chunks\\n            n = max(n, chunks)\\n        embs, pooled = [], None\\n        for i in range(n):\\n            parts = []\\n            for tok, te, ids in ((self.tok1, self.te1, ids1), (self.tok2, self.te2, ids2)):\\n                c = ids[i * 75:(i + 1) * 75]\\n                pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id\\n                row = [tok.bos_token_id] + c + [tok.eos_token_id]\\n                row = row + [pad] * (77 - len(row))\\n                o = te(torch.tensor([row], device=self.dev), output_hidden_states=True)\\n                parts.append(o.hidden_states[-2])\\n                if te is self.te2 and i == 0:\\n                    pooled = o.text_embeds\\n            embs.append(torch.cat(parts, -1))\\n        return torch.cat(embs, 1), pooled, n\\n\\n    # -------------------------------------------------------------- loras\\n    def set_loras(self, loras: list[dict]):\\n        \\"\\"\\"Activate LoRA files ({\\"path\\", \\"scale\\"}); others are disabled.\\"\\"\\"\\n        names = []\\n        for l in loras:\\n            name = os.path.splitext(os.path.basename(l[\\"path\\"]))[0].replace(\\".\\", \\"_\\")\\n            if name not in self.loaded_loras:\\n                from diffusers.loaders.lora_pipeline import StableDiffusionXLLoraLoaderMixin as M\\n                sd, alphas = M.lora_state_dict(l[\\"path\\"])\\n                M.load_lora_into_unet(sd, alphas, self.unet, adapter_name=name)\\n                self.loaded_loras[name] = l[\\"path\\"]\\n            names.append((name, l.get(\\"scale\\", 0.8)))\\n        if names:\\n            self.unet.set_adapters([n for n, _ in names], [s for _, s in names])\\n        elif self.loaded_loras:\\n            self.unet.disable_lora()\\n\\n    # -------------------------------------------------------------- vae\\n    @torch.no_grad()\\n    def decode(self, lat: torch.Tensor) -> Image.Image:\\n        x = self.vae.decode(lat.float().to(self.dev) / self.vae.config.scaling_factor).sample.cpu()\\n        x = (x[0].clamp(-1, 1) + 1) / 2\\n        return Image.fromarray((x.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8))\\n\\n    @torch.no_grad()\\n    def encode_image(self, img: Image.Image) -> torch.Tensor:\\n        x = (torch.from_numpy(np.asarray(img.convert(\\"RGB\\"))).float().permute(2, 0, 1)[None] / 127.5 - 1).to(self.dev)\\n        return self.vae.encode(x).latent_dist.mean * self.vae.config.scaling_factor\\n\\n    # -------------------------------------------------------------- sampling\\n    @torch.no_grad()\\n    def sample(self, prompt: str, negative: str, width: int, height: int, steps: int = 26, cfg: float = 6.0,\\n               seed: int = 0, controls: dict | None = None, control_end: float = 0.75,\\n               regions: list[tuple[str, np.ndarray]] | None = None, base_weight: float = 0.3,\\n               init_image: Image.Image | None = None, strength: float = 1.0,\\n               inpaint_mask: np.ndarray | None = None, references: list | None = None,\\n               style_fidelity: float = 0.5, progress: str = \\"\\") -> Image.Image:\\n        \\"\\"\\"controls: {\\"openpose\\": (PIL, scale), \\"depth\\": (PIL, scale), ...}\\n        regions: [(prompt, mask HxW float)] - masks need not sum to 1; leftover -> base prompt only.\\n        inpaint_mask: HxW float, 1 = regenerate (only with init_image).\\n        references: [Reference] identity anchors (character sheets), each restricted to its mask.\\"\\"\\"\\n        from diffusers import DPMSolverMultistepScheduler\\n\\n        sched = DPMSolverMultistepScheduler.from_config(self.sched_config, use_karras_sigmas=True,\\n                                                        algorithm_type=\\"dpmsolver++\\", solver_order=2,\\n                                                        final_sigmas_type=\\"zero\\", timestep_spacing=\\"trailing\\")\\n        sched.set_timesteps(steps, device=self.dev)\\n        timesteps = sched.timesteps\\n        if init_image is not None and strength < 1.0:\\n            start = int(round(steps * (1 - strength)))\\n            timesteps = timesteps[start:]\\n            if hasattr(sched, \\"set_begin_index\\"):\\n                sched.set_begin_index(start)\\n        g = torch.Generator().manual_seed(seed)\\n        h8, w8 = height // 8, width // 8\\n\\n        # prompts (all padded to the same number of 77-token chunks)\\n        lens = [self.encode(p)[2] for p in [prompt, negative] + [r[0] for r in (regions or [])]]\\n        n = max(lens)\\n        pos, pooled, _ = self.encode(prompt, n)\\n        neg, npooled, _ = self.encode(negative, n)\\n        ctx = torch.cat([neg, pos]).to(self.dtype)\\n        add_text = torch.cat([npooled, pooled]).to(self.dtype)\\n        tids = torch.tensor([[height, width, 0, 0, height, width]] * 2, dtype=self.dtype, device=self.dev)\\n        added = {\\"text_embeds\\": add_text, \\"time_ids\\": tids}\\n\\n        st = self.region\\n        st.cache.clear()\\n        if regions:\\n            st.contexts = [self.encode(p, n)[0].to(self.dtype) for p, _ in regions]\\n            ms = np.stack([np.asarray(Image.fromarray((m * 255).astype(np.uint8)).resize((w8, h8), Image.BILINEAR),\\n                                      np.float32) / 255 for _, m in regions])\\n            st.masks = torch.from_numpy(ms).to(self.dev)\\n            st.base_weight = base_weight\\n            st.active = True\\n        else:\\n            st.active, st.contexts = False, []\\n\\n        # controls\\n        cn_in = None\\n        if controls and self.controlnet is not None:\\n            imgs, modes, scales = [], [], []\\n            for kind, (im, scale) in controls.items():\\n                im = im.convert(\\"RGB\\").resize((width, height), Image.BILINEAR)\\n                t = torch.from_numpy(np.array(im)).float().permute(2, 0, 1)[None] / 255\\n                imgs.append(torch.cat([t, t]).to(self.dev, self.dtype))\\n                modes.append(CONTROL_MODES[kind])\\n                scales.append(float(scale))\\n            ctype = torch.zeros(self.controlnet.config.num_control_type)\\n            ctype[modes] = 1\\n            cn_in = dict(controlnet_cond=imgs, control_type=ctype[None].repeat(2, 1).to(self.dev, self.dtype),\\n                         control_type_idx=modes, conditioning_scale=scales)\\n\\n        # references: encode once; every step they are re-noised to t and run through the UNet\\n        refs = []\\n        rs = self.ref\\n        rs.cache.clear()\\n        rs.banks, rs.mode = [], \\"off\\"\\n        if references:\\n            ms = []\\n            for ref in references:\\n                rw, rh = _fit_area(ref.image.size, 0.62e6)\\n                rlat = self.encode_image(ref.image.convert(\\"RGB\\").resize((rw, rh), Image.LANCZOS))\\n                remb, rpool, _ = self.encode(ref.prompt)\\n                rids = torch.tensor([[rh, rw, 0, 0, rh, rw]], dtype=self.dtype, device=self.dev)\\n                refs.append((rlat, torch.randn(rlat.shape, generator=g).to(self.dev), remb.to(self.dtype),\\n                             {\\"text_embeds\\": rpool.to(self.dtype), \\"time_ids\\": rids}))\\n                m = np.ones((height, width), np.float32) if ref.mask is None else ref.mask.astype(np.float32)\\n                ms.append(np.asarray(Image.fromarray((np.clip(m, 0, 1) * 255).astype(np.uint8))\\n                                     .resize((w8, h8), Image.BILINEAR), np.float32) / 255)\\n                rs.banks.append({})\\n            rs.masks = torch.from_numpy(np.stack(ms)).to(self.dev)\\n            rs.weights = [r.weight for r in references]\\n            rs.style_fidelity = style_fidelity\\n\\n        # latents\\n        noise = torch.randn((1, 4, h8, w8), generator=g).to(self.dev)\\n        if init_image is not None:\\n            init = self.encode_image(init_image.resize((width, height), Image.LANCZOS))\\n            lat = sched.add_noise(init, noise, timesteps[:1]) if strength < 1.0 else noise * sched.init_noise_sigma\\n        else:\\n            init = None\\n            lat = noise * sched.init_noise_sigma\\n        keep = None\\n        if inpaint_mask is not None and init is not None:\\n            mk = Image.fromarray((np.clip(inpaint_mask, 0, 1) * 255).astype(np.uint8)).resize((w8, h8), Image.BILINEAR)\\n            keep = (1 - torch.from_numpy(np.asarray(mk, np.float32) / 255)[None, None]).to(self.dev)\\n\\n        t0 = time.time()\\n        total = len(timesteps)\\n        for i, t in enumerate(timesteps):\\n            if refs:\\n                st.active, was = False, st.active\\n                rs.mode = \\"write\\"\\n                for ri, (rlat, rnoise, remb, radd) in enumerate(refs):\\n                    rs.current = ri\\n                    rx = sched.add_noise(rlat, rnoise, t.reshape(1))\\n                    rx = sched.scale_model_input(rx, t).to(self.dtype)\\n                    self.unet(rx, t, encoder_hidden_states=remb, added_cond_kwargs=radd, return_dict=False)\\n                rs.mode = \\"read\\"\\n                st.active = was\\n            x = torch.cat([lat, lat])\\n            x = sched.scale_model_input(x, t).to(self.dtype)\\n            kw = {}\\n            if cn_in is not None and i < control_end * total:\\n                down, mid = self.controlnet(x, t, encoder_hidden_states=ctx, added_cond_kwargs=added,\\n                                            return_dict=False, **cn_in)\\n                kw = dict(down_block_additional_residuals=down, mid_block_additional_residual=mid)\\n            eps = self.unet(x, t, encoder_hidden_states=ctx, added_cond_kwargs=added, return_dict=False, **kw)[0]\\n            eu, ec = eps.float().chunk(2)\\n            e = eu + cfg * (ec - eu)\\n            # CFG rescale (keeps saturated anime colours from blowing out at higher cfg)\\n            std_c, std_e = ec.std(), e.std()\\n            e = 0.3 * (e * std_c / std_e) + 0.7 * e\\n            lat = sched.step(e, t, lat, return_dict=False)[0]\\n            if keep is not None:\\n                if i + 1 < total:\\n                    ref = sched.add_noise(init, noise, timesteps[i + 1:i + 2])\\n                else:\\n                    ref = init\\n                lat = lat * (1 - keep) + ref * keep\\n            el = time.time() - t0\\n            print(f\\"\\\\r[{progress}] step {i + 1}/{total}  {el / (i + 1):.1f}s/it  eta {el / (i + 1) * (total - i - 1):.0f}s\\",\\n                  end=\\"\\", flush=True)\\n        print(flush=True)\\n        st.active = False\\n        rs.mode, rs.banks = \\"off\\", []\\n        return self.decode(lat)\\n", "animescene/gpu_server.py": "\\"\\"\\"HTTP front for the diffusion Engine, to run it on a borrowed GPU (e.g. a free Colab T4).\\n\\nPOST /sample  {token, prompt, negative, width, height, steps, cfg, seed, strength, control_end, base_weight,\\n               init: png64, controls: {kind: [png64, scale]}, regions: [[prompt, png64 mask]]} -> {image: png64}\\n`RemoteEngine(url, token)` has the same `sample()` as Engine, so tools/anime_pass.py runs either way.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport base64\\nimport io\\nimport json\\nimport urllib.request\\n\\nimport numpy as np\\nfrom PIL import Image\\n\\n\\ndef _enc(im: Image.Image) -> str:\\n    b = io.BytesIO()\\n    im.save(b, \\"PNG\\")\\n    return base64.b64encode(b.getvalue()).decode()\\n\\n\\ndef _dec(s: str) -> Image.Image:\\n    return Image.open(io.BytesIO(base64.b64decode(s)))\\n\\n\\ndef _mask_enc(m: np.ndarray) -> str:\\n    return _enc(Image.fromarray((np.clip(m, 0, 1) * 255).astype(np.uint8)))\\n\\n\\nclass RemoteEngine:\\n    def __init__(self, url: str, token: str, timeout=900):\\n        self.url, self.token, self.timeout = url.rstrip(\\"/\\"), token, timeout\\n\\n    def sample(self, prompt, negative, width, height, steps=26, cfg=6.0, seed=0, controls=None, control_end=0.75,\\n               regions=None, base_weight=0.3, init_image=None, strength=1.0, progress=\\"\\", **_):\\n        body = {\\"token\\": self.token, \\"prompt\\": prompt, \\"negative\\": negative, \\"width\\": width, \\"height\\": height,\\n                \\"steps\\": steps, \\"cfg\\": cfg, \\"seed\\": seed, \\"control_end\\": control_end, \\"base_weight\\": base_weight,\\n                \\"strength\\": strength, \\"init\\": _enc(init_image) if init_image is not None else None,\\n                \\"controls\\": {k: [_enc(im), sc] for k, (im, sc) in (controls or {}).items()},\\n                \\"regions\\": [[p, _mask_enc(m)] for p, m in (regions or [])]}\\n        req = urllib.request.Request(self.url + \\"/sample\\", json.dumps(body).encode(), {\\"Content-Type\\": \\"application/json\\"})\\n        with urllib.request.urlopen(req, timeout=self.timeout) as r:\\n            return _dec(json.loads(r.read())[\\"image\\"]).convert(\\"RGB\\")\\n\\n\\ndef serve(models=\\"models\\", port=7860, token=None):\\n    import secrets\\n    import threading\\n    from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\\n\\n    from animescene.diffusion import Engine\\n    token = token or secrets.token_urlsafe(16)\\n    eng = Engine(models)\\n    lock = threading.Lock()\\n\\n    class H(BaseHTTPRequestHandler):\\n        def do_GET(self):\\n            self.send_response(200)\\n            self.end_headers()\\n            self.wfile.write(b\\"ok\\")\\n\\n        def do_POST(self):\\n            q = json.loads(self.rfile.read(int(self.headers[\\"Content-Length\\"])))\\n            if q.get(\\"token\\") != token:\\n                self.send_response(403)\\n                self.end_headers()\\n                return\\n            controls = {k: (_dec(v[0]).convert(\\"RGB\\"), float(v[1])) for k, v in q[\\"controls\\"].items()}\\n            regions = [(p, np.asarray(_dec(m).convert(\\"L\\"), np.float32) / 255) for p, m in q[\\"regions\\"]]\\n            with lock:  # one image at a time on the GPU\\n                im = eng.sample(q[\\"prompt\\"], q[\\"negative\\"], q[\\"width\\"], q[\\"height\\"], steps=q[\\"steps\\"], cfg=q[\\"cfg\\"],\\n                                seed=q[\\"seed\\"], controls=controls, control_end=q[\\"control_end\\"], regions=regions,\\n                                base_weight=q[\\"base_weight\\"], strength=q[\\"strength\\"],\\n                                init_image=_dec(q[\\"init\\"]).convert(\\"RGB\\") if q.get(\\"init\\") else None, progress=\\"remote\\")\\n            out = json.dumps({\\"image\\": _enc(im)}).encode()\\n            self.send_response(200)\\n            self.send_header(\\"Content-Type\\", \\"application/json\\")\\n            self.end_headers()\\n            self.wfile.write(out)\\n\\n    srv = ThreadingHTTPServer((\\"0.0.0.0\\", port), H)\\n    threading.Thread(target=srv.serve_forever, daemon=True).start()\\n    return token\\n", "fetch_models.py": "\\"\\"\\"Fetch the model weights into a diffusers layout.\\n\\nWeights used (all public on HuggingFace):\\n  sdxl/              Ine007/waiIllustriousSDXL_v150          (WAI-Illustrious-SDXL v15, anime SDXL)\\n  vae_fix/           madebyollin/sdxl-vae-fp16-fix\\n  controlnet_union/  xinsir/controlnet-union-sdxl-1.0 (promax)\\n\\n`--source hf` downloads them from HuggingFace (needs huggingface_hub and access to huggingface.co).\\n`--source dockerhub` is for networks where HuggingFace is blocked but Docker Hub is not: it pulls\\nthe HuggingFace-cache layers of public images that baked these exact repos in, extracts *only*\\nweight/config data (never code), and checks every LFS blob against the sha256 it is named by.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport json\\nimport os\\nimport tarfile\\nimport urllib.request\\n\\nACCEPT = (\\"application/vnd.docker.distribution.manifest.list.v2+json, application/vnd.oci.image.index.v1+json, \\"\\n          \\"application/vnd.docker.distribution.manifest.v2+json, application/vnd.oci.image.manifest.v1+json\\")\\n# (image, layer digest prefix, kind)\\nDOCKER = {\\n    \\"sdxl\\": (\\"wxderxr/wai-illustrious-v150\\", \\"368076eb0f25\\", \\"hfcache\\"),\\n    \\"vae_fix\\": (\\"wxderxr/wai-illustrious-v150\\", \\"bb88cb87f8f8\\", \\"hfcache\\"),\\n    \\"controlnet_union\\": (\\"jsmorphogen/worker-controlnet-union\\", \\"b21b11d74ca7\\", \\"dir\\"),\\n}\\nHF = {\\"sdxl\\": \\"Ine007/waiIllustriousSDXL_v150\\", \\"vae_fix\\": \\"madebyollin/sdxl-vae-fp16-fix\\",\\n      \\"controlnet_union\\": (\\"xinsir/controlnet-union-sdxl-1.0\\", \\"diffusion_pytorch_model_promax.safetensors\\")}\\nSAFE = (\\".safetensors\\", \\".json\\", \\".txt\\")\\n\\n\\ndef _token(repo):\\n    u = f\\"https://auth.docker.io/token?service=registry.docker.io&scope=repository:{repo}:pull\\"\\n    return json.load(urllib.request.urlopen(u))[\\"token\\"]\\n\\n\\ndef _get(repo, path, tok, accept=True):\\n    h = {\\"Authorization\\": \\"Bearer \\" + tok}\\n    if accept:\\n        h[\\"Accept\\"] = ACCEPT\\n    return urllib.request.urlopen(urllib.request.Request(f\\"https://registry-1.docker.io/v2/{repo}/{path}\\", headers=h))\\n\\n\\ndef _layer(repo, prefix):\\n    tok = _token(repo)\\n    m = json.load(_get(repo, \\"manifests/latest\\", tok))\\n    if \\"manifests\\" in m:\\n        d = [x for x in m[\\"manifests\\"] if x.get(\\"platform\\", {}).get(\\"architecture\\") == \\"amd64\\"][0][\\"digest\\"]\\n        m = json.load(_get(repo, f\\"manifests/{d}\\", tok))\\n    digest = [l[\\"digest\\"] for l in m[\\"layers\\"] if l[\\"digest\\"][7:].startswith(prefix)][0]\\n    return _get(repo, f\\"blobs/{digest}\\", tok, accept=False)\\n\\n\\ndef _sha256(path):\\n    h = hashlib.sha256()\\n    with open(path, \\"rb\\") as f:\\n        while b := f.read(1 << 24):\\n            h.update(b)\\n    return h.hexdigest()\\n\\n\\ndef fetch_docker(name, dest):\\n    repo, prefix, kind = DOCKER[name]\\n    out = os.path.join(dest, name)\\n    os.makedirs(out, exist_ok=True)\\n    blobs, links = {}, []\\n    with _layer(repo, prefix) as r, tarfile.open(fileobj=r, mode=\\"r|gz\\") as tf:\\n        for mem in tf:\\n            if kind == \\"hfcache\\":\\n                if mem.issym() and \\"/snapshots/\\" in mem.name:\\n                    links.append((mem.name.split(\\"/snapshots/\\")[1].split(\\"/\\", 1)[1], os.path.basename(mem.linkname)))\\n                elif mem.isfile() and \\"/blobs/\\" in mem.name:\\n                    p = os.path.join(out, \\".blobs\\", os.path.basename(mem.name))\\n                    os.makedirs(os.path.dirname(p), exist_ok=True)\\n                    src = tf.extractfile(mem)\\n                    with open(p, \\"wb\\") as f:\\n                        while chunk := src.read(1 << 24):\\n                            f.write(chunk)\\n                    blobs[os.path.basename(mem.name)] = p\\n            elif mem.isfile() and mem.name.endswith(SAFE):\\n                p = os.path.join(out, os.path.basename(mem.name))\\n                src = tf.extractfile(mem)\\n                with open(p, \\"wb\\") as f:\\n                    while chunk := src.read(1 << 24):\\n                        f.write(chunk)\\n                print(\\"  \\", p)\\n    for rel, blob in links:\\n        if not rel.endswith(SAFE):\\n            continue\\n        if len(blob) == 64 and _sha256(blobs[blob]) != blob:\\n            raise SystemExit(f\\"checksum mismatch for {rel}\\")\\n        p = os.path.join(out, rel)\\n        os.makedirs(os.path.dirname(p), exist_ok=True)\\n        os.replace(blobs[blob], p) if not os.path.exists(p) else None\\n        print(\\"  \\", p)\\n\\n\\ndef fetch_hf(name, dest):\\n    from huggingface_hub import hf_hub_download, snapshot_download\\n    out = os.path.join(dest, name)\\n    src = HF[name]\\n    if isinstance(src, tuple):\\n        repo, fname = src\\n        os.makedirs(out, exist_ok=True)\\n        hf_hub_download(repo, \\"config_promax.json\\", local_dir=out)\\n        hf_hub_download(repo, fname, local_dir=out)\\n        os.replace(os.path.join(out, \\"config_promax.json\\"), os.path.join(out, \\"config.json\\"))\\n        os.replace(os.path.join(out, fname), os.path.join(out, \\"diffusion_pytorch_model.safetensors\\"))\\n    else:\\n        snapshot_download(src, local_dir=out, allow_patterns=[\\"*.json\\", \\"*.txt\\", \\"*/*.safetensors\\", \\"diffusion_pytorch_model*.safetensors\\"],\\n                          ignore_patterns=[\\"*fp32*\\", \\"*.bin\\"])\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"--dest\\", default=os.environ.get(\\"ANIMESCENE_MODELS\\", \\"models\\"))\\n    ap.add_argument(\\"--source\\", choices=[\\"hf\\", \\"dockerhub\\"], default=\\"hf\\")\\n    a = ap.parse_args()\\n    for name in DOCKER:\\n        print(f\\"[{name}]\\")\\n        (fetch_hf if a.source == \\"hf\\" else fetch_docker)(name, a.dest)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
for p, src in FILES.items():
    os.makedirs(os.path.dirname(p) or '.', exist_ok=True)
    open(p, 'w').write(src)
print('wrote', list(FILES))

In [ ]:
!python fetch_models.py --dest models --source hf

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

In [ ]:
import subprocess, re, time
from animescene.gpu_server import serve
token = serve('models', 7860)
p = subprocess.Popen(['./cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:7860'],
                     stderr=subprocess.PIPE, text=True)
url = None
while url is None:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', p.stderr.readline())
    url = m and m.group(0)
print('URL  :', url)
print('TOKEN:', token)

In [ ]:
# keeps the session busy so Colab doesn't idle it out while panels render
import time
while True:
    time.sleep(60)